In [1]:
import os
import json
import math
import re
import sqlite3
import time
from collections import Counter, defaultdict
from typing import Any, TypedDict

import numpy as np
from openai import OpenAI




In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook: export OPENAI_API_KEY=sk-...")

client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")

print(f"OpenAI ready. Chat model: {CHAT_MODEL}; embedding model: {EMBED_MODEL}")

OpenAI ready. Chat model: gpt-4o-mini; embedding model: text-embedding-3-small


In [3]:
CORPUS = [
    {
        "id": "pods",
        "source": "pods.md",
        "text": "A Kubernetes Pod is the smallest deployable unit. Containers in a Pod share network, storage volumes, and lifecycle. Use kubectl describe pod and kubectl logs to debug Pod behavior.",
    },
    {
        "id": "deployment",
        "source": "deployment.md",
        "text": "A Deployment manages ReplicaSets and rolling updates. Use kubectl rollout status deployment/nginx to watch progress and kubectl rollout undo deployment/nginx to roll back a bad release.",
    },
    {
        "id": "service",
        "source": "service.md",
        "text": "A Service gives stable networking for Pods. ClusterIP is internal, NodePort exposes a port on nodes, and LoadBalancer asks the cloud provider for an external endpoint.",
    },
    {
        "id": "probes",
        "source": "probes.md",
        "text": "Readiness probes decide when a Pod can receive traffic. Liveness probes restart stuck containers. Startup probes protect slow-starting applications from premature restarts.",
    },
    {
        "id": "secrets",
        "source": "secrets.md",
        "text": "Kubernetes Secrets store sensitive values such as passwords, tokens, and keys. Enable encryption at rest, restrict RBAC, and avoid printing secret data in logs.",
    },
    {
        "id": "taints",
        "source": "taints.md",
        "text": "Taints repel Pods from nodes. Tolerations allow selected Pods to schedule onto tainted nodes. A NoSchedule taint blocks Pods that lack a matching toleration.",
    },
    {
        "id": "hpa",
        "source": "hpa.md",
        "text": "The HorizontalPodAutoscaler increases or decreases replicas based on CPU, memory, or custom metrics so an application can handle more traffic automatically.",
    },
]

NOISE_DOCS = [
    {"id": "noise-cache", "source": "cache-paper.md", "text": "Cache-aware matrix multiplication improves locality in CPU memory hierarchies and reduces cache misses."},
    {"id": "noise-graph", "source": "graph-paper.md", "text": "Graph partitioning algorithms optimize edge cuts in distributed computation workloads."},
]

print(f"Inline corpus loaded: {len(CORPUS)} K8s snippets + {len(NOISE_DOCS)} noise snippets")

Inline corpus loaded: 7 K8s snippets + 2 noise snippets


In [4]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


def cosine(a: list[float], b: list[float]) -> float:
    av = np.array(a, dtype=float)
    bv = np.array(b, dtype=float)
    denom = np.linalg.norm(av) * np.linalg.norm(bv)
    return float(np.dot(av, bv) / denom) if denom else 0.0


def dense_search(question: str, docs: list[dict[str, str]], top_k: int = 3) -> list[dict[str, Any]]:
    vectors = embed_texts([question] + [d["text"] for d in docs])
    qv, doc_vectors = vectors[0], vectors[1:]
    ranked = []
    for doc, dv in zip(docs, doc_vectors):
        ranked.append({**doc, "score": cosine(qv, dv)})
    return sorted(ranked, key=lambda d: d["score"], reverse=True)[:top_k]


def answer_with_context(question: str, chunks: list[dict[str, Any]]) -> str:
    context = "\n\n".join(f"SOURCE: {c['source']}\n{c['text']}" for c in chunks)
    messages = [
        {"role": "system", "content": "Answer only from the provided Kubernetes context. Cite source names inline."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    response = client.chat.completions.create(model=CHAT_MODEL, messages=messages, temperature=0)
    return response.choices[0].message.content

In [5]:
question = "What is the best practice for managing application secrets securely?"
candidates = dense_search(question, CORPUS + NOISE_DOCS, top_k=6)
for i, c in enumerate(candidates, 1):
    print(f"{i}. dense={c['score']:.3f} {c['source']}")

1. dense=0.589 secrets.md
2. dense=0.200 deployment.md
3. dense=0.192 pods.md
4. dense=0.189 probes.md
5. dense=0.170 hpa.md
6. dense=0.140 service.md


In [6]:
VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY")
if not VOYAGE_API_KEY:
    raise RuntimeError("Set VOYAGE_API_KEY before running this notebook: export VOYAGE_API_KEY=...")

import voyageai

voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)
VOYAGE_RERANK_MODEL = "rerank-2.5"

print(f"Voyage ready. rerank model: {VOYAGE_RERANK_MODEL}")

/Users/divyansh/Work/UK_PROJECTS/Enterprise_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Voyage ready. rerank model: rerank-2.5


In [7]:
def rerank_with_voyage(question: str, candidates: list[dict[str, Any]]) -> list[dict[str, Any]]:
    documents = [c["text"] for c in candidates]
    response = voyage_client.rerank(
        query=question,
        documents=documents,
        model=VOYAGE_RERANK_MODEL,
        top_k=len(candidates),
    )

    reranked = []
    for item in response.results:
        reranked.append({**candidates[item.index], "rerank_score": item.relevance_score})
    return reranked

In [8]:
voyage_reranked = rerank_with_voyage(question, candidates)
print(f"{'rank':<5}{'source':<20}{'dense score':<15}{'voyage score':<15}")
for i, v in enumerate(voyage_reranked, 1):
    print(f"{i:<5}{v['source']:<20}{v['score']:<15.4f}{v['rerank_score']:<15.4f}")

rank source              dense score    voyage score   
1    secrets.md          0.5887         0.7461         
2    deployment.md       0.2001         0.3574         
3    pods.md             0.1920         0.3359         
4    service.md          0.1398         0.3086         
5    probes.md           0.1894         0.2695         
6    hpa.md              0.1699         0.2676         
